# Συγκριτική αποτύπωση των πειραμάτων JEPA

Το notebook συγκεντρώνει τις επαληθευμένες αξιολογήσεις της αναπτυξιακής
πορείας του JEPA στο Othello 8×8 — **Transformer και Mamba** — και δημιουργεί:

1. έναν τελικό συγκριτικό πίνακα,
2. ένα διπλό διάγραμμα με την καλύτερη διαθέσιμη top-1 ακρίβεια νόμιμης κίνησης
   δίπλα στην καλύτερη ακρίβεια σχετικής κατάστασης του ταμπλό.

Ο Mamba κωδικοποιητής εκπαιδεύτηκε μόνο στα δύο τελευταία στάδια της
αναζήτησης — hard-disjoint στόχο και hard-disjoint με σήμα εκπαίδευσης από όλες
τις έγκυρες θέσεις — καθώς και στην εκδοχή δράσης v6. Τα στάδια με κοινό
ιστορικό δεν επαναλήφθηκαν σε Mamba.

Κάθε τιμή διαβάζεται από τα artifacts της αντίστοιχης εκτέλεσης· τίποτα δεν
είναι γραμμένο με το χέρι.


In [1]:
try:
    from google.colab import drive
except ImportError:
    print("Local execution: using the existing artifact drive.")
else:
    drive.mount("/content/drive")

Local execution: using the existing artifact drive.


In [2]:
from __future__ import annotations

import json
import os
import sys
import warnings
from pathlib import Path
from typing import Any, Iterable

import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")


def find_project_root(start: Path | None = None) -> Path:
    """Locate the repository root from a notebook, script, or interactive shell."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


def resolve_artifact_root() -> Path:
    """Resolve the artifacts directory on Windows or Google Colab."""
    candidates = [
        os.environ.get("MASTER_THESIS_ARTIFACT_ROOT"),
        r"G:\My Drive\Master_Thesis_Artifacts",
        "/content/drive/MyDrive/Master_Thesis_Artifacts",
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return Path(candidate)
    raise FileNotFoundError(
        "Δεν βρέθηκε ο φάκελος Master_Thesis_Artifacts. "
        "Ορίστε τη μεταβλητή MASTER_THESIS_ARTIFACT_ROOT."
    )


PROJECT_ROOT = find_project_root()
ARTIFACT_ROOT = resolve_artifact_root()
RUNS_ROOT = ARTIFACT_ROOT / "runs"
OUTPUT_DIR = Path(
    os.environ.get(
        "JEPA_COMPARISON_OUTPUT_DIR",
        PROJECT_ROOT / "JEPA_Experimentation" / "comparison_outputs",
    )
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Artifact root:", ARTIFACT_ROOT)
print("Output dir   :", OUTPUT_DIR)



Project root : C:\Users\Babis\Dropbox\MSc_AI\Master_Thesis_Code_Final
Artifact root: G:\My Drive\Master_Thesis_Artifacts
Output dir   : C:\Users\Babis\Dropbox\MSc_AI\Master_Thesis_Code_Final\JEPA_Experimentation\comparison_outputs


## Μητρώο πειραμάτων

Η σειρά του μητρώου ακολουθεί την πραγματική ερευνητική πορεία: αρχικά οι
εκδοχές v1–v5 με κοινό ιστορικό context/target, έπειτα η v6, στη συνέχεια οι
hard-disjoint επανεκτελέσεις και, τέλος, οι αντίστοιχες εκδοχές με σήμα
εκπαίδευσης από όλες τις έγκυρες θέσεις. Μέσα σε κάθε στάδιο προηγούνται οι
εκτελέσεις Transformer και ακολουθούν οι Mamba, ώστε τα δύο υποσύνολα να
διαβάζονται χωριστά μέσα στην ίδια ζώνη.

Το πεδίο `architecture` είναι το μόνο που προστέθηκε στη δομή· οι αναγνώστες
των artifacts (`legacy`, `smoke`, `sections`, `unified`) χρησιμοποιούνται
αυτούσιοι και για τις δύο αρχιτεκτονικές.


In [3]:
EXPERIMENTS: list[dict[str, Any]] = [
    # ---- Initial objectives: target representation still depends on the history.
    dict(order=1, architecture="Transformer", phase="Κοινό ιστορικό (nested)", version="v1", objective="VICReg",
         run_paths=["jepa_linear_b8_run_001"], kind="legacy",
         legal={"linear": "eval_legal_moves__final.json"},
         board={"linear": "eval_board_state_probes__final.json"}),
    dict(order=2, architecture="Transformer", phase="Κοινό ιστορικό (nested)", version="v2", objective="Smooth L1, EMA, K=4",
         run_paths=["jepa_transformer_b8_run_001"], kind="legacy",
         legal={"linear": "eval_legal_moves__final.json"},
         board={"linear": "eval_board_state_probes__final.json"}),
    dict(order=3, architecture="Transformer", phase="Κοινό ιστορικό (nested)", version="v3", objective="Smooth L1, EMA, K=8 (collapse)",
         run_paths=["jepa_v3_mlp_k8_b8_run_001"], kind="legacy",
         legal={"linear": "eval_legal_moves__final.json"},
         board={"linear": "eval_board_state_probes__final.json"}, flag="collapse"),
    dict(order=4, architecture="Transformer", phase="Κοινό ιστορικό (nested)", version="v4", objective="Smooth L1, multi-position K=4",
         run_paths=["jepa_v4_multi_pos_mlp_k4_b8_run_001"], kind="legacy",
         legal={"linear": "eval_legal_moves__final.json"},
         board={"linear": "eval_board_state_probes__final.json"}),
    dict(order=5, architecture="Transformer", phase="Κοινό ιστορικό (nested)", version="v5", objective="InfoNCE",
         run_paths=["jepa_v5_contrastive_prefix_mask"], kind="legacy",
         legal={"linear": "eval_legal_moves__final.json"},
         board={"linear": "eval_board_state_probes__final.json",
                "mlp": "eval_board_mlp__final.json"}),

    # ---- v6 introduced the action-only target and a multi-mode predictor.
    dict(order=6, architecture="Transformer", phase="v6 hard-disjoint δράση", version="v6", objective="Distance-margin, 10 modes",
         run_paths=["jepa_v6_hard_disjoint_action_run_001"], kind="sections",
         result="chunks_10/evaluations__final_v6_hard_disjoint.json"),
    dict(order=7, architecture="Mamba", phase="v6 hard-disjoint δράση", version="v6", objective="Distance-margin, 10 modes",
         run_paths=["Mamba-JEPA/mamba_jepa_v6_hd_action"], kind="smoke",
         result="smoke_eval/smoke_eval_results__final.json"),

    # ---- Hard-disjoint reruns, one sampled training boundary per game.
    dict(order=8, architecture="Transformer", phase="Hard-disjoint", version="v1", objective="VICReg",
         run_paths=["jepa_vicreg_b8_run_001_hard_disjoint_bf16"], kind="smoke",
         result="smoke_eval/smoke_eval_results_normalized_v2__final.json"),
    dict(order=9, architecture="Transformer", phase="Hard-disjoint", version="v2", objective="Smooth L1, EMA, K=4",
         run_paths=["jepa_transformer_b8_run_001_hard_disjoint"], kind="smoke",
         result="smoke_eval/smoke_eval_results__final.json"),
    dict(order=10, architecture="Transformer", phase="Hard-disjoint", version="v4", objective="Smooth L1, multi-position K=4",
         run_paths=["jepa_v4_multi_pos_mlp_k4_b8_run_001_hard_disjoint_vicreg"], kind="smoke",
         result="smoke_eval/smoke_eval_results__final.json"),
    dict(order=11, architecture="Transformer", phase="Hard-disjoint", version="v5", objective="InfoNCE",
         run_paths=["jepa_v5_contrastive_prefix_mask_hard_disjoint"], kind="sections",
         result="chunks_200/evaluations__final_v5_hard_disjoint.json"),
    dict(order=12, architecture="Mamba", phase="Hard-disjoint", version="v1", objective="VICReg",
         run_paths=["Mamba-JEPA/mamba_jepa_v1_hd_vicreg"], kind="smoke",
         result="smoke_eval/smoke_eval_results_normalized_v2__final.json"),
    dict(order=13, architecture="Mamba", phase="Hard-disjoint", version="v5", objective="InfoNCE",
         run_paths=["Mamba-JEPA/mamba_jepa_v5_hd_infonce"], kind="unified",
         result="unified_eval/final/results__unified_eval_v2.json"),

    # ---- Same objectives with all-position training signal.
    dict(order=14, architecture="Transformer", phase="Hard-disjoint + all-position", version="v1", objective="VICReg",
         run_paths=["jepa_vicreg_b8_run_001_hard_disjoint_allpos_bf16"], kind="unified",
         result="unified_eval/final/results__unified_eval_v2.json"),
    dict(order=15, architecture="Transformer", phase="Hard-disjoint + all-position", version="v2", objective="Smooth L1, EMA, K=4",
         run_paths=["jepa_v2_ema_b8_run_001_hd_allpos_bf16"], kind="unified",
         result="unified_eval/final/results__unified_eval_v2.json"),
    dict(order=16, architecture="Transformer", phase="Hard-disjoint + all-position", version="v4", objective="Smooth L1, multi-position K=4",
         run_paths=["jepa_v4_multi_pos_k4_hd_run_001_allpos_bf16"], kind="unified",
         result="unified_eval/final/results__unified_eval_v2.json"),
    dict(order=17, architecture="Transformer", phase="Hard-disjoint + all-position", version="v5", objective="InfoNCE",
         run_paths=[
             "Transformer-JEPA/jepa_v5_infonce_b8_run_001_hd_allpos_bf16",
             "jepa_v5_infonce_b8_run_001_hd_allpos_bf16",
         ], kind="unified", result="unified_eval/final/results__unified_eval_v2.json"),
    dict(order=18, architecture="Mamba", phase="Hard-disjoint + all-position", version="v1", objective="VICReg",
         run_paths=["Mamba-JEPA/mamba_jepa_v1_hd_vicreg_allpos_b8"], kind="unified",
         result="unified_eval/final/results__unified_eval_v2.json"),
    dict(order=19, architecture="Mamba", phase="Hard-disjoint + all-position", version="v1e", objective="VICReg + EMA",
         run_paths=["Mamba-JEPA/mamba_jepa_v1_hd_vicreg_ema_allpos_b8"], kind="unified",
         result="unified_eval/final/results__unified_eval_v2.json"),
    dict(order=20, architecture="Mamba", phase="Hard-disjoint + all-position", version="v5", objective="InfoNCE",
         run_paths=["Mamba-JEPA/mamba_jepa_v5_hd_infonce_allpos_b8"], kind="unified",
         result="unified_eval/final/results__unified_eval_v2.json"),
]


def read_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def resolve_run(spec: dict[str, Any]) -> Path | None:
    for relative in spec["run_paths"]:
        path = RUNS_ROOT / relative
        if path.exists():
            return path
    return None


def numeric_layer_items(container: dict[str, Any]) -> Iterable[tuple[int, dict[str, Any]]]:
    for key, value in container.items():
        try:
            layer = int(key)
        except (TypeError, ValueError):
            continue
        if isinstance(value, dict):
            yield layer, value


def relative_accuracy(layer_value: dict[str, Any]) -> float | None:
    """Read a relative-board accuracy from the legacy evaluation schemas."""
    candidates = [
        layer_value.get("test", {}).get("relative", {}).get("accuracy"),
        layer_value.get("val", {}).get("relative", {}).get("accuracy"),
        layer_value.get("relative", {}).get("accuracy"),
    ]
    return next((float(value) for value in candidates if value is not None), None)


def best_board_from_layer_container(
    container: dict[str, Any], probe: str
) -> dict[str, Any] | None:
    candidates: list[dict[str, Any]] = []
    for layer, value in numeric_layer_items(container):
        accuracy = relative_accuracy(value)
        if accuracy is not None:
            candidates.append({"accuracy": accuracy, "probe": probe, "layer": layer})
    return max(candidates, key=lambda item: item["accuracy"], default=None)


def best_board_from_payload(payload: dict[str, Any], probe: str) -> dict[str, Any] | None:
    """Support legacy, smoke-eval, and section-based board-probe payloads."""
    containers: list[dict[str, Any]] = []
    for candidate in (
        payload,
        payload.get("metrics"),
        payload.get("results"),
        payload.get("results", {}).get("metrics") if isinstance(payload.get("results"), dict) else None,
    ):
        if isinstance(candidate, dict):
            containers.append(candidate)

    found = [best_board_from_layer_container(container, probe) for container in containers]
    found = [item for item in found if item is not None]
    return max(found, key=lambda item: item["accuracy"], default=None)


def choose_best(items: Iterable[dict[str, Any] | None], key: str) -> dict[str, Any] | None:
    valid = [item for item in items if item is not None and item.get(key) is not None]
    return max(valid, key=lambda item: item[key], default=None)


def protocol_name(value: Any, fallback: str) -> str:
    if isinstance(value, str):
        return value
    if isinstance(value, dict):
        for key in ("tag", "name", "version", "protocol"):
            if value.get(key):
                return str(value[key])
    return fallback


def dataframe_to_markdown(frame: pd.DataFrame) -> str:
    """Small dependency-free Markdown table writer."""
    text = frame.fillna("—").astype(str)
    headers = [str(column).replace("|", "\\|") for column in text.columns]
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for values in text.itertuples(index=False, name=None):
        escaped = [value.replace("|", "\\|").replace("\n", " ") for value in values]
        lines.append("| " + " | ".join(escaped) + " |")
    return "\n".join(lines) + "\n"


def next_move_integrity(payloads: dict[str, Any]) -> str | None:
    """Reject a legal-move evaluation whose numbers cannot be real.

    A NaN in the encoder output makes ``argmax`` return the same index for
    every position, so both readouts report identical metrics and the
    probability mass is serialised as ``null``.  The resulting top-1 is the
    frequency with which one constant square happens to be legal — an artifact
    of the failure, not a measurement of the model.
    """
    metrics = {}
    for readout, payload in payloads.items():
        metrics[readout] = payload.get("metrics", payload)

    finite_mass = [m.get("legal_prob_mass_per_token") for m in metrics.values()]
    if metrics and all(value is None for value in finite_mass):
        return ("μη πεπερασμένα logits: legal_prob_mass_per_token = null "
                "σε όλους τους readouts")

    if len(metrics) > 1:
        curves = [json.dumps(m.get("top1_legal_per_position")) for m in metrics.values()]
        if len(set(curves)) == 1:
            return "ταυτόσημες μετρικές linear και mlp: ένα μόνο σύνολο προβλέψεων"

    for readout, m in metrics.items():
        top1 = m.get("top1_legal_per_token")
        topk1 = (m.get("topk_legal") or {}).get("1")
        if top1 is not None and topk1 is not None and abs(float(top1) - float(topk1)) > 0.01:
            return (f"ασυνεπές {readout}: top1={float(top1):.4f} ενώ "
                    f"topk[1]={float(topk1):.4f}")
    return None


def parse_legacy(spec: dict[str, Any], run_dir: Path) -> dict[str, Any]:
    legal_items = []
    eval_games = []
    source_paths = []
    for readout, relative_path in spec["legal"].items():
        path = run_dir / relative_path
        data = read_json(path)
        source_paths.append(path)
        value = data.get("top1_legal_per_token")
        if value is not None:
            legal_items.append({"accuracy": float(value), "readout": readout})
        if data.get("n_games") is not None:
            eval_games.append(int(data["n_games"]))

    board_items = []
    for probe, relative_path in spec["board"].items():
        path = run_dir / relative_path
        data = read_json(path)
        source_paths.append(path)
        board_items.append(best_board_from_payload(data, probe))

    metadata_path = run_dir / "eval_metadata__final.json"
    metadata = read_json(metadata_path) if metadata_path.exists() else {}
    games_seen = metadata.get("games_seen")
    if games_seen is None:
        metrics_path = run_dir / "metrics.csv"
        if metrics_path.exists():
            metrics = pd.read_csv(metrics_path)
            if "games_seen" in metrics.columns and not metrics.empty:
                games_seen = int(metrics["games_seen"].dropna().iloc[-1])

    return dict(
        legal=choose_best(legal_items, "accuracy"),
        board=choose_best(board_items, "accuracy"),
        games_seen=games_seen,
        eval_games=max(eval_games) if eval_games else None,
        protocol="legacy evaluation",
        source_paths=source_paths,
    )


def parse_smoke(spec: dict[str, Any], run_dir: Path) -> dict[str, Any]:
    path = run_dir / spec["result"]
    data = read_json(path)
    problem = next_move_integrity(data.get("next_move_legal", {}))
    if problem is not None:
        raise ValueError(f"{problem} ({path.name})")
    legal_items = []
    eval_games = []
    for readout, payload in data.get("next_move_legal", {}).items():
        metrics = payload.get("metrics", payload)
        value = metrics.get("top1_legal_per_token")
        if value is not None:
            legal_items.append({"accuracy": float(value), "readout": readout})
        if metrics.get("n_games") is not None:
            eval_games.append(int(metrics["n_games"]))

    board_items = []
    for probe in ("linear", "mlp"):
        payload = data.get("board_state", {}).get(probe)
        if isinstance(payload, dict):
            board_items.append(best_board_from_payload(payload, probe))

    return dict(
        legal=choose_best(legal_items, "accuracy"),
        board=choose_best(board_items, "accuracy"),
        games_seen=data.get("metadata", {}).get("games_seen"),
        eval_games=max(eval_games) if eval_games else None,
        protocol=protocol_name(
            data.get("metadata", {}).get("board_probe_protocol"),
            "smoke evaluation",
        ),
        source_paths=[path],
    )


def parse_sections(spec: dict[str, Any], run_dir: Path) -> dict[str, Any]:
    path = run_dir / spec["result"]
    data = read_json(path)
    legal_items: list[dict[str, Any]] = []
    board_items: list[dict[str, Any] | None] = []
    eval_games: list[int] = []

    for section in data.get("sections", []):
        title = str(section.get("title", ""))
        title_lower = title.lower()
        metrics = section.get("metrics", {})

        # Exclude predictor-space diagnostics: the comparison concerns the
        # frozen context encoder and its downstream readouts.
        if "predictor-space" in title_lower:
            continue

        if "legal-move evaluation" in title_lower:
            readout = "mlp" if title_lower.startswith("mlp") else "linear"
            legal_metrics = metrics.get("legal_metrics", metrics)
            value = legal_metrics.get("top1_legal_per_token")
            if value is not None:
                legal_items.append({"accuracy": float(value), "readout": readout})
            if legal_metrics.get("n_games") is not None:
                eval_games.append(int(legal_metrics["n_games"]))

        if "board state probe" in title_lower or "board state probes" in title_lower:
            probe = "mlp" if title_lower.startswith("mlp") else "linear"
            board_items.append(best_board_from_payload(metrics, probe))

    return dict(
        legal=choose_best(legal_items, "accuracy"),
        board=choose_best(board_items, "accuracy"),
        games_seen=data.get("metadata", {}).get("games_seen"),
        eval_games=max(eval_games) if eval_games else None,
        protocol="development evaluation",
        source_paths=[path],
    )


def parse_unified(spec: dict[str, Any], run_dir: Path) -> dict[str, Any]:
    path = run_dir / spec["result"]
    data = read_json(path)
    legal_items = []
    eval_games = []
    for readout, payload in data.get("frozen_next_move", {}).items():
        metrics = payload.get("test_legal", {})
        value = metrics.get("top1_legal_per_token")
        if value is not None:
            legal_items.append({"accuracy": float(value), "readout": readout})
        if metrics.get("n_games") is not None:
            eval_games.append(int(metrics["n_games"]))

    board_items = []
    for probe, payload in data.get("board_state", {}).items():
        selected = payload.get("selected", {}).get("relative", {})
        layer = selected.get("layer")
        if layer is None:
            continue
        layer_metrics = payload.get("test_metrics", {}).get(str(layer), {})
        value = layer_metrics.get("relative", {}).get("accuracy")
        if value is not None:
            board_items.append({"accuracy": float(value), "probe": probe, "layer": int(layer)})

    metadata = data.get("metadata", {})
    return dict(
        legal=choose_best(legal_items, "accuracy"),
        board=choose_best(board_items, "accuracy"),
        games_seen=metadata.get("games_seen"),
        eval_games=max(eval_games) if eval_games else None,
        protocol=protocol_name(metadata.get("protocol"), "unified_eval_v2"),
        source_paths=[path],
    )


PARSERS = {
    "legacy": parse_legacy,
    "smoke": parse_smoke,
    "sections": parse_sections,
    "unified": parse_unified,
}


def build_results() -> pd.DataFrame:
    rows = []
    for spec in EXPERIMENTS:
        run_dir = resolve_run(spec)
        base = dict(
            order=spec["order"],
            architecture=spec["architecture"],
            phase=spec["phase"],
            version=spec["version"],
            objective=spec["objective"],
            run_name=run_dir.name if run_dir else spec["run_paths"][0],
            flag=spec.get("flag", ""),
        )
        if spec["kind"] == "missing" or run_dir is None:
            note = spec.get("note", "Δεν βρέθηκε ο φάκελος του run.")
            warnings.warn(f"{spec['phase']} {spec['version']}: {note}")
            rows.append({**base, "status": "missing", "note": note})
            continue

        try:
            parsed = PARSERS[spec["kind"]](spec, run_dir)
        except (FileNotFoundError, KeyError, ValueError, TypeError) as exc:
            note = f"Αποτυχία ανάγνωσης artifact: {exc}"
            warnings.warn(f"{spec['phase']} {spec['version']}: {note}")
            rows.append({**base, "status": "invalid", "note": note})
            continue

        legal = parsed["legal"] or {}
        board = parsed["board"] or {}
        rows.append({
            **base,
            "status": "ok",
            "top1_legal": legal.get("accuracy"),
            "top1_readout": legal.get("readout"),
            "board_relative_accuracy": board.get("accuracy"),
            "board_probe": board.get("probe"),
            "board_layer": board.get("layer"),
            "games_seen": parsed.get("games_seen"),
            "eval_games": parsed.get("eval_games"),
            "protocol": parsed.get("protocol"),
            "source": "; ".join(str(path) for path in parsed.get("source_paths", [])),
            "note": spec.get("note", ""),
        })

    frame = pd.DataFrame(rows).sort_values("order").reset_index(drop=True)
    return frame


results = build_results()
results



Out[1]: 
    order architecture  ...                                             source note
0       1  Transformer  ...  G:\My Drive\Master_Thesis_Artifacts\runs\jepa_...     
1       2  Transformer  ...  G:\My Drive\Master_Thesis_Artifacts\runs\jepa_...     
2       3  Transformer  ...  G:\My Drive\Master_Thesis_Artifacts\runs\jepa_...     
3       4  Transformer  ...  G:\My Drive\Master_Thesis_Artifacts\runs\jepa_...     
4       5  Transformer  ...  G:\My Drive\Master_Thesis_Artifacts\runs\jepa_...     
5       6  Transformer  ...  G:\My Drive\Master_Thesis_Artifacts\runs\jepa_...     
6       7        Mamba  ...  G:\My Drive\Master_Thesis_Artifacts\runs\Mamba...     
7       8  Transformer  ...  G:\My Drive\Master_Thesis_Artifacts\runs\jepa_...     
8       9  Transformer  ...  G:\My Drive\Master_Thesis_Artifacts\runs\jepa_...     
9      10  Transformer  ...  G:\My Drive\Master_Thesis_Artifacts\runs\jepa_...     
10     11  Transformer  ...  G:\My Drive\Master_Thesis_Artifacts\ru

## Έλεγχοι συνέπειας και τελικός πίνακας

Για τα παλαιότερα πρωτόκολλα επιλέγεται η μεγαλύτερη διαθέσιμη ακρίβεια
σχετικής κατάστασης μεταξύ των εκπαιδευμένων probes και layers. Στο
`unified_eval_v2` χρησιμοποιείται η επίδοση του test split στο layer που είχε
προηγουμένως επιλεγεί στο selection split. Μεταξύ linear και MLP εμφανίζεται η
καλύτερη διαθέσιμη τιμή, ενώ ο τύπος του readout/probe καταγράφεται ρητά.


In [4]:
required_orders = {spec["order"] for spec in EXPERIMENTS}
assert set(results["order"]) == required_orders
assert len(required_orders) == len(EXPERIMENTS)
assert results["order"].is_monotonic_increasing

valid = results[results["status"] == "ok"]
assert valid["top1_legal"].between(0, 1).all()
assert valid["board_relative_accuracy"].between(0, 1).all()

missing = results[results["status"] != "ok"]
if not missing.empty:
    print("Μη διαθέσιμα ή μη έγκυρα σημεία:")
    display(missing[["phase", "version", "run_name", "note"]])

table = results.copy()
table["Πείραμα"] = table["version"] + " — " + table["objective"]
table["Top-1 νόμιμη κίνηση"] = table["top1_legal"].map(
    lambda value: "—" if pd.isna(value) else f"{100 * value:.1f}%"
)
table["Readout"] = table["top1_readout"].fillna("—").str.upper()
table["Σχετική κατάσταση ταμπλό"] = table["board_relative_accuracy"].map(
    lambda value: "—" if pd.isna(value) else f"{100 * value:.1f}%"
)
table["Probe / layer"] = table.apply(
    lambda row: "—" if pd.isna(row.get("board_layer"))
    else f"{str(row.get('board_probe')).upper()} / L{int(row['board_layer'])}",
    axis=1,
)
table["Παιχνίδια εκπαίδευσης"] = table["games_seen"].map(
    lambda value: "—" if pd.isna(value) else f"{int(value):,}"
)
table["Παιχνίδια αξιολόγησης"] = table["eval_games"].map(
    lambda value: "—" if pd.isna(value) else f"{int(value):,}"
)

display_columns = [
    "phase", "architecture", "Πείραμα", "Top-1 νόμιμη κίνηση", "Readout",
    "Σχετική κατάσταση ταμπλό", "Probe / layer",
    "Παιχνίδια εκπαίδευσης", "Παιχνίδια αξιολόγησης", "protocol",
]
display_table = table[display_columns].rename(columns={
    "phase": "Στάδιο",
    "architecture": "Αρχιτεκτονική",
    "protocol": "Πρωτόκολλο",
})
display(display_table)

csv_path = OUTPUT_DIR / "jepa_experiments_comparison.csv"
markdown_path = OUTPUT_DIR / "jepa_experiments_comparison.md"
results.to_csv(csv_path, index=False, encoding="utf-8-sig")
markdown_path.write_text(dataframe_to_markdown(display_table), encoding="utf-8")
print("CSV     :", csv_path)
print("Markdown:", markdown_path)


def render_table_image(frame: pd.DataFrame, path: Path) -> None:
    compact = frame[[
        "Στάδιο", "Πείραμα", "Top-1 νόμιμη κίνηση",
        "Readout", "Σχετική κατάσταση ταμπλό", "Probe / layer",
    ]].copy()
    compact.columns = [
        "Στάδιο", "Πείραμα", "Top-1 legal", "Readout",
        "Σχετική κατάσταση", "Probe / layer",
    ]
    fig_height = 0.48 * len(compact) + 1.4
    fig, ax = plt.subplots(figsize=(15.5, fig_height))
    ax.axis("off")
    tbl = ax.table(
        cellText=compact.values,
        colLabels=compact.columns,
        cellLoc="center",
        colLoc="center",
        loc="center",
        colWidths=[0.20, 0.26, 0.12, 0.08, 0.14, 0.10],
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.7)
    tbl.scale(1, 1.45)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor("#D7DEE7")
        if row == 0:
            cell.set_facecolor("#243B53")
            cell.set_text_props(color="white", weight="bold")
        else:
            cell.set_facecolor("#F7F9FC" if row % 2 else "white")
            if col in (0, 1):
                cell.set_text_props(ha="left")
    ax.set_title("Συγκριτικός πίνακας αναπτυξιακών πειραμάτων JEPA (8×8)",
                 fontsize=14, fontweight="bold", pad=18, color="#102A43")
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)


table_png = OUTPUT_DIR / "jepa_experiments_comparison_table.png"
render_table_image(display_table, table_png)
print("Table PNG:", table_png)



CSV     : C:\Users\Babis\Dropbox\MSc_AI\Master_Thesis_Code_Final\JEPA_Experimentation\comparison_outputs\jepa_experiments_comparison.csv
Markdown: C:\Users\Babis\Dropbox\MSc_AI\Master_Thesis_Code_Final\JEPA_Experimentation\comparison_outputs\jepa_experiments_comparison.md
Table PNG: C:\Users\Babis\Dropbox\MSc_AI\Master_Thesis_Code_Final\JEPA_Experimentation\comparison_outputs\jepa_experiments_comparison_table.png


,Στάδιο,Αρχιτεκτονική,Πείραμα,Top-1 νόμιμη κίνηση,Readout,Σχετική κατάσταση ταμπλό,Probe / layer,Παιχνίδια εκπαίδευσης,Παιχνίδια αξιολόγησης,Πρωτόκολλο
0,Κοινό ιστορικό (nested),Transformer,v1 — VICReg,25.7%,LINEAR,62.4%,LINEAR / L0,"19,999,840","5,000",legacy evaluation
1,Κοινό ιστορικό (nested),Transformer,"v2 — Smooth L1, EMA, K=4",38.6%,LINEAR,62.0%,LINEAR / L6,"19,999,840","5,000",legacy evaluation
2,Κοινό ιστορικό (nested),Transformer,"v3 — Smooth L1, EMA, K=8 (collapse)",27.2%,LINEAR,59.7%,LINEAR / L0,"19,999,840","5,000",legacy evaluation
3,Κοινό ιστορικό (nested),Transformer,"v4 — Smooth L1, multi-position K=4",33.6%,LINEAR,62.7%,LINEAR / L1,"19,999,840","5,000",legacy evaluation
4,Κοινό ιστορικό (nested),Transformer,v5 — InfoNCE,39.6%,LINEAR,63.0%,LINEAR / L5,"19,999,840","5,000",legacy evaluation
5,v6 hard-disjoint δράση,Transformer,"v6 — Distance-margin, 10 modes",62.2%,MLP,63.1%,MLP / L0,"4,999,952","5,000",development evaluation
6,v6 hard-disjoint δράση,Mamba,"v6 — Distance-margin, 10 modes",57.1%,MLP,61.7%,MLP / L0,"19,999,840","5,000",smoke evaluation
7,Hard-disjoint,Transformer,v1 — VICReg,79.1%,MLP,70.3%,LINEAR / L5,"19,999,840","5,000",normalized_v2
8,Hard-disjoint,Transformer,"v2 — Smooth L1, EMA, K=4",37.8%,MLP,62.0%,MLP / L0,"4,999,952","5,000",smoke evaluation
9,Hard-disjoint,Transformer,"v4 — Smooth L1, multi-position K=4",25.3%,MLP,60.5%,MLP / L0,"4,999,956","5,000",smoke evaluation


## Διπλό διάγραμμα της αναπτυξιακής πορείας

Ένα σχήμα με δύο σκέλη: αριστερά η top-1 ακρίβεια νόμιμης κίνησης, δεξιά η
ακρίβεια σχετικής κατάστασης του ταμπλό. Το φόντο δηλώνει το στάδιο της
αναζήτησης, το χρώμα του σημείου την αρχιτεκτονική, ο τύπος γραμμής επίσης την
αρχιτεκτονική ώστε η ταυτότητα να μην στηρίζεται μόνο στο χρώμα, και το σχήμα
του σημείου τον readout ή probe που έδωσε την εμφανιζόμενη τιμή.


In [5]:
PHASE_COLORS = {
    "Κοινό ιστορικό (nested)": "#4C956C",
    "v6 hard-disjoint δράση": "#805AD5",
    "Hard-disjoint": "#D97706",
    "Hard-disjoint + all-position": "#2563A6",
}
PHASE_SHORT_LABELS = {
    "Κοινό ιστορικό (nested)": "Κοινό ιστορικό (nested)",
    "v6 hard-disjoint δράση": "v6",
    "Hard-disjoint": "Hard-disjoint",
    "Hard-disjoint + all-position": "HD + all-position",
}
TYPE_MARKERS = {"linear": "o", "mlp": "s"}
ARCH_COLORS = {"Transformer": "#2a78d6", "Mamba": "#eb6834"}
ARCH_LINESTYLES = {"Transformer": "-", "Mamba": "--"}


def phase_spans(frame: pd.DataFrame) -> list[tuple[float, float, str]]:
    spans = []
    for phase, group in frame.groupby("phase", sort=False):
        x = group["order"].to_numpy(dtype=float) - 1
        spans.append((x.min() - 0.48, x.max() + 0.48, phase))
    return spans


def timeline_panel(ax, frame, metric, type_column, ylabel, ymin):
    for left, right, phase in phase_spans(frame):
        ax.axvspan(left, right, color=PHASE_COLORS[phase], alpha=0.075, zorder=0)
        ax.text((left + right) / 2, 1.012, PHASE_SHORT_LABELS[phase],
                transform=ax.get_xaxis_transform(), ha="center", va="bottom",
                fontsize=7.0, fontweight="bold", color=PHASE_COLORS[phase])

    # One continuous line per architecture; points without a valid value are
    # skipped so the line bridges them instead of breaking.
    for architecture, group in frame.groupby("architecture", sort=False):
        group = group.sort_values("order").dropna(subset=[metric])
        if len(group) < 2:
            continue
        ax.plot(group["order"].to_numpy(dtype=float) - 1,
                group[metric].to_numpy(dtype=float),
                color=ARCH_COLORS[architecture],
                linestyle=ARCH_LINESTYLES[architecture],
                linewidth=1.5, alpha=0.55, zorder=1)

    for _, row in frame.iterrows():
        x = row["order"] - 1
        value = row.get(metric)
        color = ARCH_COLORS[row["architecture"]]
        if pd.isna(value):
            missing_y = ymin + 0.035 * (1.0 - ymin)
            ax.scatter(x, missing_y, marker="x", s=55, color="#9B2C2C", zorder=4)
            ax.annotate("άκυρο\nartifact", (x, missing_y), xytext=(0, 7),
                        textcoords="offset points", ha="center", fontsize=6.2,
                        color="#9B2C2C")
            continue
        marker = TYPE_MARKERS.get(str(row.get(type_column, "linear")).lower(), "D")
        ax.scatter(x, value, marker=marker, s=62, color=color,
                   edgecolor="white", linewidth=1.0, zorder=3)
        ax.annotate(f"{100 * value:.1f}", (x, value), xytext=(0, 7),
                    textcoords="offset points", ha="center", fontsize=6.6,
                    color="#243B53")
        if row.get("flag") == "collapse":
            ax.annotate("collapse", (x, value), xytext=(0, -15),
                        textcoords="offset points", ha="center", fontsize=6.6,
                        color="#9B2C2C")

    x_all = frame["order"].to_numpy(dtype=float) - 1
    ax.set_xticks(x_all, [row.version for row in frame.itertuples()], fontsize=7.0)
    ax.set_xlabel("Διαδοχή πειραμάτων ανά στάδιο", fontsize=9.0)
    ax.set_ylim(ymin, 1.06)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_ylabel(ylabel, fontsize=9.5)
    ax.grid(axis="y", color="#D9E2EC", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)


fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(13.6, 5.8))

timeline_panel(ax_left, results, "top1_legal", "top1_readout",
               "Top-1 ακρίβεια νόμιμης κίνησης", ymin=0.0)
ax_left.set_title("Top-1 ακρίβεια νόμιμης κίνησης", fontsize=11.5,
                  fontweight="bold", color="#102A43", pad=26)

timeline_panel(ax_right, results, "board_relative_accuracy", "board_probe",
               "Ακρίβεια σχετικής κατάστασης ταμπλό", ymin=0.45)
ax_right.set_title("Ακρίβεια σχετικής κατάστασης του ταμπλό", fontsize=11.5,
                   fontweight="bold", color="#102A43", pad=26)

arch_handles = [
    Line2D([0], [0], color=ARCH_COLORS[name], linestyle=ARCH_LINESTYLES[name],
           marker="o", markersize=6, markeredgecolor="white", linewidth=1.6, label=name)
    for name in ARCH_COLORS
]
type_handles = [
    Line2D([0], [0], marker=marker, color="#526D82", linestyle="none",
           markersize=7, label=f"readout / probe: {name.upper()}")
    for name, marker in TYPE_MARKERS.items()
]
fig.legend(handles=arch_handles + type_handles, ncol=4, loc="lower center",
           bbox_to_anchor=(0.5, 0.045), frameon=False, fontsize=8.0)

fig.suptitle("Εξέλιξη της αναπτυξιακής πορείας JEPA στο Othello 8×8",
             fontsize=13.5, fontweight="bold", color="#102A43", y=0.995)
fig.text(
    0.01, 0.005,
    "Σημείωση: Οι τιμές είναι ποσοστά. Η v3 εμφανίζεται μόνο στο αρχικό στάδιο· μετά την "
    "κατάρρευση των αναπαραστάσεων δεν περιλαμβάνεται στα επόμενα στάδια του διαγράμματος. "
    "Ο Mamba δεν εκτελέστηκε στα στάδια με κοινό ιστορικό.\n"
    "Η χρονογραμμή είναι περιγραφική: τα στάδια δεν έχουν όλα ίδιο budget και πρωτόκολλο "
    "αξιολόγησης, και οι τιμές των πρώιμων σταδίων προέρχονται από μικρότερες αξιολογήσεις.",
    fontsize=7.0, color="#526D82",
)

fig.tight_layout(rect=(0, 0.105, 1, 0.975))
combined_path = OUTPUT_DIR / "jepa_timeline_combined.png"
fig.savefig(combined_path, dpi=300, bbox_inches="tight", facecolor="white")
if "ipykernel" in sys.modules:
    plt.show()
plt.close(fig)

print("Combined plot:", combined_path)


Combined plot: C:\Users\Babis\Dropbox\MSc_AI\Master_Thesis_Code_Final\JEPA_Experimentation\comparison_outputs\jepa_timeline_combined.png


## Σύντομη αριθμητική σύνοψη


In [6]:
best_top1 = valid.loc[valid["top1_legal"].idxmax()]
best_board = valid.loc[valid["board_relative_accuracy"].idxmax()]
print(
    f"Καλύτερο top-1 legal: {best_top1['phase']} {best_top1['version']} "
    f"({best_top1['objective']}), {best_top1['top1_legal']:.2%} "
    f"με {str(best_top1['top1_readout']).upper()} readout."
)
print(
    f"Καλύτερη σχετική κατάσταση: {best_board['phase']} {best_board['version']} "
    f"({best_board['objective']}), {best_board['board_relative_accuracy']:.2%} "
    f"με {str(best_board['board_probe']).upper()} probe στο L{int(best_board['board_layer'])}."
)


Καλύτερο top-1 legal: Hard-disjoint + all-position v5 (InfoNCE), 99.72% με LINEAR readout.
Καλύτερη σχετική κατάσταση: Hard-disjoint + all-position v5 (InfoNCE), 98.17% με LINEAR probe στο L12.
